# LLM 모델링 실험 노트북
**담당자:** 김동건  
**모델:** Groq (llama-3.3-70b-versatile)  
**목적:** 고인의 디지털 기록을 LLM으로 분석하여 유족을 위한 보고서 자동 생성


## 0. 환경 설정

In [ ]:
import os
import sys
import json
from pathlib import Path
from dotenv import load_dotenv
from groq import Groq

# 프로젝트 루트 경로 추가
sys.path.append(str(Path().resolve().parent))

load_dotenv('../.env')
client = Groq(api_key=os.getenv('GROQ_API_KEY'))
MODEL = 'llama-3.3-70b-versatile'

print('환경 설정 완료')
print(f'모델: {MODEL}')

## 1. 데이터 로드 및 전처리

In [ ]:
from utils.preprocessing import preprocess_all

data = preprocess_all()
print('\n로드된 카테고리:', list(data['llm_input'].keys()))

In [ ]:
# 카테고리별 데이터 샘플 확인
for cat, text in data['llm_input'].items():
    print(f'\n[{cat}] 앞 200자:')
    print(text[:200])
    print('-' * 50)

## 2. 프롬프트 실험
### 2-1. 구독 서비스 분석

In [ ]:
subscription_text = data['llm_input'].get('subscription', '')

prompt_v1 = f"""
아래는 고인의 구독 서비스 결제 알림 내역입니다.
각 구독 서비스별로 정리하여 유족이 해지해야 할 목록을 JSON으로 반환해주세요.
JSON만 반환하고 다른 텍스트는 쓰지 마세요.

[결제 내역]
{subscription_text}

[출력 형식]
{{
  "미해지_구독_목록": [
    {{
      "서비스명": "Netflix",
      "월결제금액": 17000,
      "해지방법": "앱 또는 홈페이지에서 해지",
      "우선순위": "높음"
    }}
  ],
  "예상_월간_지출": 0,
  "총_구독수": 0
}}
"""

response = client.chat.completions.create(
    model=MODEL,
    messages=[{"role": "user", "content": prompt_v1}],
    temperature=0.2,
)
result_v1 = response.choices[0].message.content
print('[구독 분석 결과]')
print(result_v1)

In [ ]:
# JSON 파싱 확인
import json

def parse_json(text):
    text = text.strip()
    if text.startswith('```'):
        text = text.split('```')[1]
        if text.startswith('json'):
            text = text[4:]
    return json.loads(text.strip())

parsed = parse_json(result_v1)
print('총 구독수:', parsed.get('총_구독수'))
print('예상 월간 지출:', parsed.get('예상_월간_지출'), '원')
for item in parsed.get('미해지_구독_목록', []):
    print(f"  - {item['서비스명']}: {item['월결제금액']:,}원 ({item['우선순위']})")

### 2-2. 보험 분석

In [ ]:
insurance_text = data['llm_input'].get('insurance', '')

prompt_insurance = f"""
아래는 고인의 보험 관련 이메일 내역입니다.
유족이 청구할 수 있는 보험금과 유지 중인 보험 목록을 JSON으로 반환해주세요.
JSON만 반환하고 다른 텍스트는 쓰지 마세요.

[보험 내역]
{insurance_text}

[출력 형식]
{{
  "보험_목록": [
    {{
      "보험사": "삼성생명",
      "보험종류": "종신보험",
      "증권번호": "1234567",
      "월납입료": 50000,
      "청구가능여부": true,
      "고객센터": "1588-0000"
    }}
  ],
  "총_보험수": 0,
  "청구가능_보험수": 0
}}
"""

response = client.chat.completions.create(
    model=MODEL,
    messages=[{"role": "user", "content": prompt_insurance}],
    temperature=0.2,
)
result_insurance = response.choices[0].message.content
print('[보험 분석 결과]')
print(result_insurance[:500])

### 2-3. 의료 기록 분석

In [ ]:
medical_text = data['llm_input'].get('medical', '')

prompt_medical = f"""
아래는 고인의 의료 관련 알림 내역입니다.
복약 중이던 약물, 주요 진단 내용, 담당 병원을 JSON으로 정리해주세요.
JSON만 반환하고 다른 텍스트는 쓰지 마세요.

[의료 내역]
{medical_text}

[출력 형식]
{{
  "복약_목록": [{{
      "약물명": "암로디핀 5mg",
      "용도": "혈압약",
      "복용시간": "오전 8시"
  }}],
  "주요_진단": ["고혈압"],
  "담당_병원": ["서울아산병원"]
}}
"""

response = client.chat.completions.create(
    model=MODEL,
    messages=[{"role": "user", "content": prompt_medical}],
    temperature=0.2,
)
print('[의료 분석 결과]')
print(response.choices[0].message.content)

## 3. 전체 파이프라인 실행

In [ ]:
from modules.llm_module import run_llm_pipeline

result = run_llm_pipeline(data)
print('분석 완료!')
print('결과 키:', list(result.keys()))

In [ ]:
# 종합 보고서 출력
from IPython.display import Markdown
Markdown(result['report'])

## 4. 결과 저장

In [ ]:
output_path = Path('../data/processed/llm_result.json')
output_path.parent.mkdir(parents=True, exist_ok=True)

with open(output_path, 'w', encoding='utf-8') as f:
    json.dump(result, f, ensure_ascii=False, indent=2)

print(f'결과 저장 완료: {output_path}')